# Fisher-Controlled Extremal Probes — Numerical Demo

Reproduces toy experiments: vertex localization, gap vs $\varepsilon$, componentwise probes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from categorical_polytope import (
    VertexProbeAlgorithm,
    FisherPrunedVertexSearch,
    robustness_sweep,
    assess_decomposition,
    build_block_fisher,
    BlockLayout,
)

In [ ]:
# Toy 2-block: gap vs Fisher coupling
couplings = [0.0, 0.05, 0.1, 0.15, 0.25, 0.35]
eps_list, gap_list, robust_list = [], [], []
for c in couplings:
    r = assess_decomposition(
        build_block_fisher(BlockLayout(('A','B'), (2,2)), off_diag_coupling=c),
        (1.0, 0.5, 2.0, 3.0),
    )
    eps_list.append(r.leakage.epsilon)
    gap_list.append(r.bounds.objective_gap_observed)
    robust_list.append(r.coproduct_robust)
list(zip(couplings, eps_list, gap_list, robust_list))

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.plot(eps_list, gap_list, 'o-', label='gap (joint - separable)')
    plt.xlabel('normalized leakage epsilon')
    plt.ylabel('objective gap')
    plt.title('Theorem 2: Phi(epsilon) regime')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
except ImportError:
    print('matplotlib not installed; see experiments/results.json')

## Non-quadratic extension (empirical Fisher)

When interaction is `face_bowl`, the maximum need not lie on a vertex — compare `grid_maximize` vs `vertex_maximize`.

In [ ]:
from categorical_polytope import NonlinearStudy

for mode, strength in [("bilinear", 0.5), ("face_bowl", 1.5)]:
    r = NonlinearStudy().analyze(strength=strength, interaction=mode)
    print(mode, "strength", strength)
    print("  vertex", r.theta_vertex.as_corner_tuple(), "value", r.value_vertex)
    print("  gap_vs_grid", r.gap_vs_grid, "vertex_ok", r.localization_at_vertex)

In [ ]:
probe = VertexProbeAlgorithm(cross_info_bound=0.25).find_near_optimal_probe()
pruned = FisherPrunedVertexSearch(fisher_epsilon=0.25, top_k=4).run()
print('vertex search:', probe.theta, probe.objective_value)
print('Fisher-pruned:', pruned.theta, pruned.objective_value, 'certified', pruned.certified)